# C-band Luna OVA queue - several fibre array placements

Generates a LabExT experiment queue that walks the fibre array across **several
placements** on the chip. Each placement gets its own move + Search for Peak, and then
however many Luna OVA measurements are read out at that placement.

Use this when the devices you want are not all reachable from one fibre array position.
Compared to the single-alignment notebook, the only difference is that alignment steps
repeat - the switch handling is identical.

**Setup assumed here**

- Dicon GP800 switch, N ports 1-7 wired to fibre array channels 1-7
- Fibre array channel 4 is the input into the chip
- The Luna OVA sits on switch M ports 3 and 4

In [1]:
# --- Hardware map -------------------------------------------------------------------
# Dicon GP800 switch: M ports are the instrument side, N ports the fibre-array side.
# N ports 1-7 are wired to fibre array channels 1-7.
FIBRE_ARRAY_CHANNELS = [1, 2, 3, 4, 5, 6, 7]

# Fibre array channel 4 launches light into the chip.
INPUT_CHANNEL = 4

# The Luna OVA occupies switch M ports 3 and 4: M3 carries the OVA source into the chip
# input, M4 returns the device output to the OVA. Swap these two if your OVA source is
# patched to M4 instead - nothing else in this notebook needs to change.
OVA_SOURCE_M_PORT = 3
OVA_RECEIVE_M_PORT = 4

# Measurement class name, as listed in the LabExT Experiment Wizard.
MEASUREMENT_CLASS = "LUNA_sweep_Cband_switch"

# Must match the chip currently imported in LabExT; device ids are looked up on it.
CHIP_NAME = "MyChip"

In [2]:
# --- OVA sweep settings -------------------------------------------------------------
# Applied to every measurement in this queue; override per measurement if needed.
# Any parameter left out here keeps the measurement class's own default.
OVA_SETTINGS = {
    "center wavelength": 1566.72,                            # nm, C band
    "wavelength range": "85.79",                            # nm, one of the OVA's fixed spans
    "Measurement Type": "Transmission",
    "Plot Measurement Type": "MIN_MAX_LOSS",
    "DUT L": 0.0,                                           # m
    "enable averaging": True,
    "number of averages": 10,
    "save_all_data": False,
    "filepath": "C:\\Users\\Luna\\Documents\\test.txt",
}

In [3]:
# --- Queue helpers ------------------------------------------------------------------
def switch_ports(output_channel):
    """M-port -> N-port routing that reads `output_channel` on the Luna OVA.

    The OVA source always drives the chip input channel; the OVA receiver follows the
    device's output channel. M1/M2 have no instrument attached, but the matrix switch
    cannot place two M ports on the same N port, so they are parked on unused channels.
    """
    if output_channel not in FIBRE_ARRAY_CHANNELS:
        raise ValueError(
            f"channel {output_channel} is not a fibre array channel {FIBRE_ARRAY_CHANNELS}"
        )
    if output_channel == INPUT_CHANNEL:
        raise ValueError(
            f"channel {output_channel} is the input channel, it cannot also be an output"
        )

    ports = {OVA_SOURCE_M_PORT: INPUT_CHANNEL, OVA_RECEIVE_M_PORT: output_channel}
    spare = [c for c in FIBRE_ARRAY_CHANNELS if c not in ports.values()]
    for m_port in (1, 2, 3, 4):
        if m_port not in ports:
            ports[m_port] = spare.pop(0)
    return {f"Switch Port: M = {m}": n for m, n in sorted(ports.items())}


def move(device_id):
    """Move the stages to a device."""
    return {"type": "move", "device_id": str(device_id)}


def sfp():
    """Run a Search for Peak at the current position."""
    return {"type": "sfp"}


def luna_meas(device_id, output_channel, **overrides):
    """One C-band Luna OVA measurement, with the switch oriented for `output_channel`."""
    parameters = dict(OVA_SETTINGS)
    parameters.update(switch_ports(output_channel))
    parameters.update(overrides)
    return {
        "type": "meas",
        "device_id": str(device_id),
        "measurement": MEASUREMENT_CLASS,
        "parameters": parameters,
    }

## Placements

One entry per fibre array position. `alignment_device` is what the stages move to and
search for peak on; `devices` maps each **output channel** to the device read out on it at
that placement.

Devices inside one placement must share an input location (they are measured at a single
alignment). Devices in *different* placements are free to sit anywhere, since each
placement re-aligns.

In [4]:
PLACEMENTS = [
    {
        "alignment_device": "1042",
        "devices": {          # output channel -> device id
            1: "1042",
            2: "1043",
            3: "1044",
        },
    },
    {
        "alignment_device": "2001",
        "devices": {
            5: "2001",
            6: "2002",
            7: "2003",
        },
    },
]

In [5]:
# --- Build the queue ----------------------------------------------------------------
entries = []

for placement in PLACEMENTS:
    entries.append(move(placement["alignment_device"]))
    entries.append(sfp())
    for output_channel, device_id in sorted(placement["devices"].items()):
        entries.append(luna_meas(device_id, output_channel))

In [6]:
# --- Preview ------------------------------------------------------------------------
# Shows the queue the way LabExT will execute it. Measurements listed under one
# alignment step form a "block": LabExT requires every device in a block to sit at the
# same input location, since the stages are only aligned once for the whole block.
for i, entry in enumerate(entries):
    if entry["type"] == "move":
        print(f"{i:3d}  MOVE -> device {entry['device_id']}")
    elif entry["type"] == "sfp":
        print(f"{i:3d}  SEARCH FOR PEAK")
    else:
        p = entry["parameters"]
        routing = " ".join(f"M{m}=N{p[f'Switch Port: M = {m}']}" for m in (1, 2, 3, 4))
        print(f"{i:3d}      meas device {entry['device_id']:>6}  [{routing}]")

  0  MOVE -> device 1042
  1  SEARCH FOR PEAK
  2      meas device   1042  [M1=N2 M2=N3 M3=N4 M4=N1]
  3      meas device   1043  [M1=N1 M2=N3 M3=N4 M4=N2]
  4      meas device   1044  [M1=N1 M2=N2 M3=N4 M4=N3]
  5  MOVE -> device 2001
  6  SEARCH FOR PEAK
  7      meas device   2001  [M1=N1 M2=N2 M3=N4 M4=N5]
  8      meas device   2002  [M1=N1 M2=N2 M3=N4 M4=N6]
  9      meas device   2003  [M1=N1 M2=N2 M3=N4 M4=N7]


In [7]:
# --- Write the queue file -----------------------------------------------------------
import json

queue = {"labext_queue_version": 1, "chip_name": CHIP_NAME, "entries": entries}

out_path = "cband_luna_multi_placement_queue.json"
with open(out_path, "w") as f:
    json.dump(queue, f, indent=2)

print(f"Wrote {len(entries)} entries to {out_path}")
print("Load it in LabExT via File -> Load Experiment Queue...")

Wrote 10 entries to cband_luna_multi_placement_queue.json
Load it in LabExT via File -> Load Experiment Queue...
